In [ ]:
import json
import time

from araclar import baslik_yazdir 

DOSYA_ADI = 'envanter.json'

class Urun:
    def __init__(self, ad, kategori, fiyat, stok):
        self.ad = ad
        self.kategori = kategori
        self.fiyat = fiyat
        self.stok = stok
        
        if self.kategori.lower() == "taze gıda" or self.kategori.lower() == "taze gida" or self.kategori.lower() == "taze gida " or self.kategori.lower() == "taze gıda ":
            self.birim = "KG"
        else:
            self.birim = "Adet"

    def json_icin_hazirla(self):
        return {
            "ad": self.ad, 
            "kategori": self.kategori, 
            "fiyat": self.fiyat, 
            "stok": self.stok,
            "birim": self.birim
        }

class Depo:
    def __init__(self):
        self.urunler = [] 
        self.verileri_yukle()

    def verileri_yukle(self):
        try:
            with open(DOSYA_ADI, 'r', encoding='utf-8') as dosya:
                kayitli_veriler = json.load(dosya)
                for veri in kayitli_veriler:
                    self.urunler.append(Urun(veri['ad'], veri['kategori'], veri['fiyat'], veri['stok']))
        except FileNotFoundError:
            print("Envanter dosyası bulunamadı. Yeni bir dosya oluşturulacak...")
            time.sleep(1)

    def verileri_kaydet(self):
        with open(DOSYA_ADI, 'w', encoding='utf-8') as dosya:
            liste_hali = [urun.json_icin_hazirla() for urun in self.urunler]
            json.dump(liste_hali, dosya, ensure_ascii=False, indent=4)

    def urun_ekle(self, ad, kategori, fiyat, stok):
        yeni_urun = Urun(ad, kategori, fiyat, stok)
        self.urunler.append(yeni_urun)
        self.verileri_kaydet()
        print(f"\n '{ad}' başarıyla depoya eklendi! ({stok} {yeni_urun.birim})")

    def listele(self):
        if not self.urunler:
            print("\n Depo şu an tamamen boş.")
            return
        
        print(f"\n{'Ürün Adı':<15} | {'Kategori':<12} | {'Fiyat':<8} | {'Stok':<12}")
        print("-" * 55)
        for u in self.urunler:
            stok_metni = f"{u.stok} {u.birim}"
            print(f"{u.ad:<15} | {u.kategori:<12} | {u.fiyat:<8.2f} | {stok_metni:<12}")

    def stok_dus(self, urun_adi, miktar):
        for u in self.urunler:
            if u.ad.lower() == urun_adi.lower():
                if u.stok >= miktar:
                    u.stok -= miktar
                    self.verileri_kaydet()
                    print(f"\n Satış başarılı! {miktar} {u.birim} '{urun_adi}' satıldı. Kalan stok: {u.stok} {u.birim}")
                else:
                    print(f"\n HATA: Yeterli stok yok! Mevcut stok sadece: {u.stok} {u.birim}")
                return
        print(f"\n HATA: Depoda '{urun_adi}' adında bir ürün bulunamadı.")

def programi_baslat():
    depo = Depo()

    while True:
        baslik_yazdir("DEPO & STOK SİSTEMİ")
        print("1. Yeni Ürün Ekle")
        print("2. Mevcut Ürünleri Listele")
        print("3. Ürün Satışı Yap (Stoktan Düş)")
        print("4. Çıkış")
        print("-" * 50)

        secim = input("Lütfen yapmak istediğiniz işlemi seçin (1-4): ")

        if secim == '1':
            print("\n--- YENİ ÜRÜN EKLE ---")
            ad = input("Ürün Adı: ")
            kategori = input("Kategori (Örn: Elektronik, Taze Gıda): ")
            try:
                fiyat = float(input("Fiyatı (Örn: 25.50): "))
                if kategori.lower() == "taze gıda" or kategori.lower() == "taze gida" :
                    stok = float(input("Stok Miktarı (KG cinsinden, Örn: 2.5): "))
                else:
                    stok = int(input("Stok Adedi: "))
                    
                depo.urun_ekle(ad, kategori, fiyat, stok)
                input("\n Menüye dönmek için ENTER tuşuna basın...")
            except ValueError:
                print("HATA: Lütfen fiyat ve stok için sadece sayı giriniz (Ondalık için nokta kullanın)!")
                input("\n Menüye dönmek için ENTER tuşuna basın...")

        elif secim == '2':
            print("\n--- ÜRÜN LİSTESİ ---")
            depo.listele()
            input("\n Listeyi inceledikten sonra menüye dönmek için ENTER tuşuna basın...")

        elif secim == '3':
            print("\n--- ÜRÜN SATIŞI ---")
            depo.listele()
            print("-" * 50)
            urun_adi = input("Satılacak Ürünün Adı: ")
            try:
                miktar = float(input("Kaç Adet / KG satıldı? (Örn: 2 veya 1.5): "))
                depo.stok_dus(urun_adi, miktar)
                input("\n Menüye dönmek için ENTER tuşuna basın...")
            except ValueError:
                print("HATA: Lütfen miktar kısmına sadece sayı giriniz!")
                input("\n Menüye dönmek için ENTER tuşuna basın...")

        elif secim == '4':
            print("\n Sistemden çıkılıyor. İyi çalışmalar!\n")
            break
        
        else:
            print("Geçersiz bir tuşa bastınız, lütfen 1 ile 4 arasında bir seçim yapın.")

programi_baslat()



★★★★★★★★★★★★★★ DEPO & STOK SİSTEMİ ★★★★★★★★★★★★★★★
1. Yeni Ürün Ekle
2. Mevcut Ürünleri Listele
3. Ürün Satışı Yap (Stoktan Düş)
4. Çıkış
--------------------------------------------------

--- ÜRÜN LİSTESİ ---

Ürün Adı        | Kategori     | Fiyat    | Stok        
-------------------------------------------------------
Makarna         | Kuru Gıda    | 50.00    | 120 Adet    
Gofret          | Atıştırmalık | 25.50    | 58 Adet     
Meyve Suyu      | Atıştırmalık | 32.00    | 75 Adet     
Elma            | Taze Gıda    | 25.00    | 25 KG       
Armut           | Taze Gıda    | 55.00    | 15 KG       
Muz             | Taze Gıda    | 70.00    | 17.0 KG     
Şeftali         | Taze Gıda    | 120.00   | 15 KG       
Nektari         | Taze Gıda    | 90.00    | 10 KG       
Erik            | Taze Gıda    | 200.00   | 5 KG        
Çilek           | Taze Gıda    | 160.00   | 5.0 KG      
Cips            | Atıştırmalık | 55.00    | 80 Adet     
Mantı           | Kuru Gıda    | 35.00    | 46